# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided, step-by-step template for loading and exploring a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access the metadata object
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's enumerate available record sets and fields, referencing all by their `@id` values.

In [ ]:
# List all record sets and their fields with @id
print("All record sets available in this dataset:")
for record_set in metadata.record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name if hasattr(record_set, 'name') else ''}")
    print(f"  Description: {record_set.description if hasattr(record_set, 'description') else ''}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}")
        print(f"      Name: {field.name if hasattr(field, 'name') else ''}")
        print(f"      Data Type: {field.data_type if hasattr(field, 'data_type') else ''}")
    print('---')

Here we explore a sample of records from the first record set, referencing its `@id`.

In [ ]:
# Show a handful of records from the first available record set
if metadata.record_sets:
    record_set_id = metadata.record_sets[0].id
    print(f"Sample records from RecordSet @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        pprint(record)
        if i >= 4:
            break
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. Each DataFrame will use the record set's `@id` as its key in the collection. Reference all record set and field IDs as defined previously.

In [ ]:
# Extract all available record sets into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded RecordSet @id: {record_set_id}")
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

# For illustration, pick the first non-empty record set for further analysis
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"Selected RecordSet @id for EDA: {main_record_set_id}")
    print(f"Fields: {dataframes[main_record_set_id].columns.tolist()}")
else:
    print("No main record set found with data.")

## 4. Exploratory Data Analysis (EDA)
Let's apply data processing and analysis steps. We'll:

- Select a numeric field by its `@id` for filtering and normalization.
- Demonstrate filtering, normalization, and grouping operations referencing fields by their `@id`s.

**Note:** Check your DataFrame for available field (column) names (which correspond to field `@id` values) and adjust accordingly.

In [ ]:
import numpy as np

# Choose a numeric field by column name (@id from the fields of the main record set)
# Example: use 'cr:MolecularWeight' if available, otherwise pick a relevant one
numeric_fields = [c for c in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][c].dtype in [np.int64, np.float64]]
if not numeric_fields:
    # Try to convert likely numeric fields
    df = dataframes[main_record_set_id]
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    numeric_fields = [c for c in df.columns if df[c].dtype in [np.int64, np.float64]]

if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Analyzing numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].dtype in [np.int64, np.float64] else 0
    
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical field if present
    # E.g., search for string/object dtype with few unique values
    group_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 10]
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        print(grouped_df)
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric fields found for EDA in the selected record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and its normalized variant.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(12, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # If normalized column was created
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(12, 5))
        sns.histplot(filtered_df[norm_col], kde=True)
        plt.title(f"Distribution of Normalized {numeric_field}")
        plt.xlabel(norm_col)
        plt.show()
else:
    print("Visualization skipped: No numeric field found.")

## 6. Conclusion
This notebook demonstrated loading, exploring, filtering, and visualizing a mass spectrometry dataset on human mast cell extracellular vesicles using Croissant schemas and the `mlcroissant` library.

- All entities and fields are referenced with their `@id` for clarity and reproducibility.
- Data extraction and filtering steps are flexible to adjust for specific record set or field availability.
  
Further exploration could include in-depth domain analysis and statistical comparisons, or joining with external protein annotations. For detailed schema and provenance, refer to the FAIR² Croissant JSON-LD linked at the top.